<a href="https://colab.research.google.com/github/letiBri/MaskArchitectureAnomaly_CourseProject/blob/main/eomt/STEP5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

In [ ]:
!git clone https://github.com/letiBri/MaskArchitectureAnomaly_CourseProject.git

%cd MaskArchitectureAnomaly_CourseProject/eomt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys # Allows importing from the 'eomt' folder
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eomt')

In [ ]:
!python3 -m pip install -r requirements.txt

In [ ]:
!pip install wandb -qU
import wandb
wandb.login()

from lightning.pytorch.loggers import WandbLogger

## Setup

In [ ]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint

seed_everything(0, verbose=False)

device = 0  # TODO: change to the GPU you want to use
IGNORE_INDEX = 255
img_idx = 10  # TODO: change to the index of the image you want to visualize
data_path = "/content/drive/MyDrive/CourseProjectAnomaly"  # drive folder of the cityscapes val set

with open("configs/dinov2/cityscapes/semantic/eomt_base_640.yaml", "r") as f:
    config_cs = yaml.safe_load(f)

with open("configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml", "r") as f:
    config_coco = yaml.safe_load(f)


def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

## Load dataset

Ensure the dataset files are correctly prepared and placed in the folder specified by `data_path`.

In [ ]:
current_config = config_coco
state_dict_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin"
target_img_size = (640, 640)
num_classes_final = 19  # classi di cityscapes

data_module_name, class_name = config_cs["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config_cs["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=2,
    num_workers=2, ## modificato
    check_empty_targets=True,
    img_size = target_img_size,
    **data_module_kwargs
)
data.setup()

## Load model

In [ ]:
warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = current_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=target_img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = current_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=num_classes_final,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = current_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in current_config["model"]["init_args"].items() if k != "network"}

if "stuff_classes" in current_config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = current_config["data"]["init_args"]["stuff_classes"]

## Load pre-trained weights from Hugging Face Hub
The model weights are downloaded from the Hugging Face Hub using the logger name from the config. Make sure you have a working internet connection.

In [ ]:
# Ripuliamo i kwargs new
model_kwargs_final = {k: v for k, v in current_config["model"]["init_args"].items() if k != "network"}
model_kwargs_final.pop("num_classes", None) # Rimuoviamo per non avere sovrascritture
model_kwargs_final.pop("img_size", None)

# FIX CRUCIALE: Rimuoviamo eventuali 'num_classes' dai kwargs che potrebbero sovrascrivere il nostro valore
# if "num_classes" in model_kwargs_final:
#   del model_kwargs_final["num_classes"]

# new
if "stuff_classes" in current_config["data"].get("init_args", {}):
    model_kwargs_final["stuff_classes"] = current_config["data"]["init_args"]["stuff_classes"]

name = current_config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")
is_dinov3 = "dinov3" in name if name else False

if is_dinov3:
    model_kwargs["ckpt_path"] = state_dict_path
    model_kwargs["delta_weights"] = True

model = (
    lit_cls(
        img_size=target_img_size,
        num_classes=num_classes_final,
        network=network,
        **model_kwargs_final,
    )
    .to(device)
)

## Transfer Learning and Fine Tuning

In [ ]:
if not is_dinov3:
    state_dict = torch.load(
        state_dict_path, map_location=f"cuda:{device}", weights_only=True
    )

    # Freeze all, only unlock the head
    keys_to_delete = [
              "network.class_head.weight",
              "network.class_head.bias",
              "criterion.empty_weight"
          ]
    for key in keys_to_delete:
        if key in state_dict:
            del state_dict[key]

model.load_state_dict(state_dict, strict=False)


In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.network.class_head.parameters():
    param.requires_grad = True

# checkpoint
checkpoint_callback = ModelCheckpoint(
  dirpath="/content/drive/MyDrive/CourseProjectAnomaly/checkpoints", # folder for saving
  filename="coco-finetuned-cityscapes-{epoch:02d}", # file name
  save_last=True, # always save last epoch
  save_top_k=1,
  monitor="metrics/val_iou_all",
  mode="max"
)

#wandb initialization
wandb_logger = WandbLogger(
    project="Anomaly_Cityscapes_Finetuning",
    name="coco_to_cityscapes_head_only"
)

# fine tuning dei parametri sbloccati
trainer = L.Trainer(
    max_epochs=3,              # 10 Numero epoche suggerito
    accelerator="gpu",


    limit_train_batches=0.05, # Usa solo il 5% dei dati
    limit_val_batches=0.05,   # Valida solo sul 5%


    devices=1,
    precision="16-mixed",       # tip del progetto
    callbacks=[checkpoint_callback],
    log_every_n_steps=10,
    logger=wandb_logger,
    default_root_dir="/content/drive/MyDrive/CourseProjectAnomaly/checkpoints"
)

# Avvio Fine-tuning
trainer.fit(model, datamodule=data)

In [ ]:
### aggiunta per inviare i risultati visuali alla dashboard online

# 1. Recupera un'immagine dal set di validazione
img_val, target_val = data.val_dataloader().dataset[img_idx]

# 2. Esegui l'inferenza usando la funzione definita prima
pred_array, target_array = infer_semantic(img_val, target_val)

# 3. Prepara le versioni colorate per la visualizzazione
mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)
pred_colored = apply_colormap(pred_array, mapping)
target_colored = apply_colormap(target_array, mapping)

# 4. Denormalizza l'immagine originale per WandB (se necessario)
# Questo serve per evitare che l'immagine originale appaia troppo scura o strana
img_input = img_val.permute(1, 2, 0).cpu().numpy()
if img_input.max() > 1.0 or img_input.min() < 0:
    img_input = (img_input - img_input.min()) / (img_input.max() - img_input.min())

# 5. Logga finalmente su WandB
wandb_logger.log_image(
    key="final_results",
    images=[img_input, pred_colored, target_colored],
    caption=["Original Image", "Model Prediction", "Ground Truth"]
)

## Semantic inference (pixel-wise classification)

> This inference method also works when applied to a model trained for panoptic segmentation.

Semantic inference computes per-pixel class scores by combining mask and class predictions:

$$
\sum_i p_i(c) \cdot m_i[h, w]
$$

Here, $p_i(c)$ is the class probability for class $c$ (excluding "no object"), and $m_i[h, w]$ is the sigmoid-normalized mask value for query $i$ at pixel $(h, w)$. The final class is selected by taking the argmax over classes.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
def infer_semantic(img, target):
    model.eval()
    model.to(device) # aggiunto per riportare in GPU
    model.network.encoder.to(device) # aggiunto per riportare in GPU

    # aggiunto Sposta l'immagine sul dispositivo del modello (GPU)
    img = img.to(device)

    # FIX  PER LE FINESTRE
    # Forza il modello a usare la dimensione della finestra corretta
    # (640 per COCO, 1024 per Cityscapes)
    model.window_size = target_img_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], target_img_size, mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()

    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()
    return pred_array, target_array


def plot_semantic_results(img, pred_array, target_array):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Gestione dell'immagine (se normalizzata o meno)
    img_np = img.permute(1, 2, 0).cpu().numpy()
    # Se i valori sono fuori dal range [0,1], normalizziamo per la visualizzazione
    if img_np.max() > 1.0 or img_np.min() < 0:
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

    axes[0].imshow(img.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title("Image")
    axes[1].imshow(apply_colormap(pred_array, mapping))
    axes[1].set_title("Prediction")
    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# run for the cityscapes pre-trained model
img_cs, target_cs = data.val_dataloader().dataset[img_idx]
pred_array_cs, target_array_cs = infer_semantic(img_cs, target_cs)
plot_semantic_results(img_cs, pred_array_cs, target_array_cs)

In [ ]:
# run for the coco pre-trained model
img_cc, target_cc = data.val_dataloader().dataset[img_idx]
pred_array_cc, target_array_cc = infer_semantic(img_cc, target_cc)
plot_semantic_results(img_cc, pred_array_cc, target_array_cc)

## Evaluation
Consistent evaluation strategy that allows fair comparison across both models for the semantic segmentation task

In [ ]:
import numpy as np
from tqdm import tqdm

def fast_hist(a, b, n):
    """
    Calcola la matrice di confusione.
    a: ground truth (target), b: predizione, n: numero di classi
    """
    k = (a >= 0) & (a < n) & (b >= 0) & (b < n) # Modifica qui: filtra anche b
    return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)

def per_class_iu(hist):
    """Calcola la IoU per ogni classe dalla matrice di confusione."""
    return np.diag(hist) / (hist.sum(1) + hist.sum(0) - np.diag(hist))

# setting
num_eval_classes = 19 # Cityscapes ha 19 classi di valutazione
hist = np.zeros((num_eval_classes, num_eval_classes))
val_loader = data.val_dataloader()

print(f"evaluation on {len(val_loader)} images...")
print(f"Current model: {'COCO' if use_coco else 'Cityscapes'}")

# evaluation, modificato per batch_size > 1
for i, batch in enumerate(tqdm(val_loader)):
    # Estraiamo le liste di immagini e target dal batch
    # batch[0] è la lista delle immagini, batch[1] è la lista dei target
    imgs_in_batch = batch[0]
    targets_in_batch = batch[1]

    # Iteriamo su ogni singolo elemento all'interno del batch (nel tuo caso, 2 iterazioni)
    for j in range(len(imgs_in_batch)):
        img = imgs_in_batch[j]
        target = targets_in_batch[j]

        # Eseguiamo l'inferenza semantica (include la mappatura se use_coco=True)
        pred_array, target_array = infer_semantic(img, target)

        # Aggiorniamo la matrice di confusione
        hist += fast_hist(target_array.flatten(), pred_array.flatten(), num_eval_classes)

# calcolo finale
ious = per_class_iu(hist)
miou = np.nanmean(ious)

# visualization results
classes = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle"
]

print("\n" + "="*30)
print(f"Results MIOU - {'COCO' if use_coco else 'CITYSCAPES'}")
print("="*30)
for name, iou in zip(classes, ious):
    print(f"{name:15}: {iou*100:6.2f}%")
print("-" * 30)
print(f"MEAN IoU: {miou*100:6.2f}%")
print("="*30)